# ODI to Databricks Migration

## `HR.TRG_LOC` Conversion

**Conversion Timestamp:** 2024-07-30 12:00:00

**Description:** This notebook converts the ODI session for loading `HR.TRG_LOC` into a Databricks Spark SQL compatible format. It includes boilerplate for ETL parameters, followed by the main data loading logic and validation steps.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "FULL", "ETL Job Type (FULL/INCREMENTAL)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "0", "ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "0", "ODI Session Number")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  ${DATASOURCE_NUM_ID} AS datasource_num_id,
  ${ETL_PROC_WID} AS etl_proc_wid,
  '${ODI_SESS_NO}' AS odi_sess_no;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_parameters"))

## Load Target Table

In [ ]:
%sql
-- SCEN_TASK_NO in {10}
-- SCEN_TASK_NO in {20}
-- SCEN_TASK_NO in {30}
INSERT INTO workspace.hr.trg_loc
(
  LOCATION_ID ,
  STREET_ADDRESS ,
  POSTAL_CODE ,
  CITY ,
  STATE_PROVINCE ,
  COUNTRY_ID
)
SELECT
  LOCATIONS.LOCATION_ID ,
  LOCATIONS.STREET_ADDRESS ,
  LOCATIONS.POSTAL_CODE ,
  LOCATIONS.CITY ,
  LOCATIONS.STATE_PROVINCE ,
  LOCATIONS.COUNTRY_ID
FROM
  workspace.hr.locations AS LOCATIONS;

## Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_rows FROM workspace.hr.trg_loc;

In [ ]:
%sql
SELECT * FROM workspace.hr.trg_loc LIMIT 10;

## Cleanup

In [ ]:
%sql
-- No temporary tables created by this specific snippet, so no explicit cleanup needed here.

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Names:** All schema names (`HR`) have been converted to lowercase and prefixed with `workspace.` (`workspace.hr`). Table names (`TRG_LOC`, `LOCATIONS`) have been converted to lowercase.
2.  **Oracle Hints:** The Oracle hint `/*+ APPEND PARALLEL */` has been removed as it is not applicable to Delta Lake in Databricks.
3.  **Data Types:** The DDL for `HR.TRG_LOC` and `HR.LOCATIONS` was not provided in the snippet. Default Spark SQL data types (e.g., `STRING` for `VARCHAR2`, `BIGINT` for `NUMBER(p,0)`) are assumed based on standard conversions. **Manual verification of target table DDL (`workspace.hr.trg_loc`) is required to ensure column names and types match expectations and the source `HR.LOCATIONS` table.**
4.  **SCEN_TASK_NO:** ODI `SCEN_TASK_NO` entries are preserved as comments in the relevant SQL cells.
5.  **Error Handling & PK Violation:** This snippet does not include logic for error table (E$) or `SNP_CHECK_TAB` as these were not present in the provided ODI SQL. If the original ODI process had these, they should be added following the guidelines in the system prompt.
6.  **Incremental/Full Load Logic:** The original snippet is a simple `INSERT`. If this was part of an incremental load in ODI, the logic for determining new/updated records (e.g., `WHERE` clauses based on `LAST_EXTRACT_TIME`) would need to be re-introduced as a `MERGE` statement.
7.  **Widgets:** Standard ETL parameter widgets (`ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `ETL_PROC_WID`, `ODI_SESS_NO`) have been included. Ensure these are set correctly when running the notebook.